In [ ]:
from core.data_sources import CLOBDataSource
from core.data_sources.hummingbot_database import HummingbotDatabase
from datetime import datetime
import sys
import os
import logging
import numpy as np
import pandas as pd
import plotly.express as px

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)


logging.getLogger("asyncio").setLevel(logging.CRITICAL)
logging.getLogger("pandas").setLevel(logging.CRITICAL)

db_names = [path for path in os.listdir(os.path.join(root_path, "data", "live_bot_databases", "brigado_server")) if path != ".gitignore"]
dbs = []
for db_name in db_names:
    if db_name.endswith(".sqlite"):
        db = HummingbotDatabase(db_name=db_name, server_name="brigado_server", root_path=root_path)
        dbs.append(db)

In [ ]:
stats = []

for db in dbs:
    if db.status["trade_fill"] == "Correct":
        # Trades
        trades_df = db.get_trade_fills()
        trades_df["date"] = pd.to_datetime(trades_df["timestamp"]).dt.strftime("%Y-%m-%d")
        first_row = trades_df.iloc[0]

        # Controllers
        controller_df = db.get_controller_data()
        if len(controller_df) == 0:
            config = None
        else:
            if len(controller_df) > 1:
                print(f"{db.db_name}: Found {len(controller_df)} controllers, only keeping first config found. Please develop multicontroller :P")
            config = controller_df["config"][0]

        stats_dict = {
            "config_file_path": first_row["config_file_path"],
            "exchange": first_row["market"],
            "trading_pair": first_row["symbol"],
            "daily_quote_volume": trades_df.groupby("date")["amount"].sum().to_dict(),
            "total_volume_usdt": trades_df["amount"].sum(),
            "pnl_usdt": trades_df["net_realized_pnl"].iloc[-1],
            "config": config,
        }
        stats.append(stats_dict)

print(f"We found problems in the following databases: {[db.db_name for db in dbs if db.status["trade_fill"] != "Correct"]}")
stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
df_expanded = (
    stats_df
    .set_index(["config_file_path", "exchange", "trading_pair"])
    ["daily_quote_volume"]
    .apply(pd.Series)
    .stack()
    .reset_index()
    .rename(columns={"level_3": "date", 0: "total_usdt_volume"})
)

df_expanded


In [ ]:
exchanges = list(df_expanded["exchange"].unique())
trading_pairs = list(df_expanded["trading_pair"].unique())
interval = "1d"
start_date = df_expanded["date"].min()
days = (datetime.now() - pd.to_datetime(start_date)).days

exchange_data = {exchange: [] for exchange in exchanges}
for exchange in exchanges:
    clob = CLOBDataSource()
    candles = await clob.get_candles_batch_last_days(connector_name=exchange,
                                                     trading_pairs=trading_pairs,
                                                     interval=interval,
                                                     days=days)
    for trading_pair in trading_pairs:
        candles_df = [candle.data for candle in candles if candle.trading_pair == trading_pair][0].copy()
        candles_df["date"] = pd.to_datetime(candles_df["timestamp"], unit="s").dt.strftime("%Y-%m-%d")
        exchange_data[exchange].append(
            {
                "candles_df": candles_df,
                "activity": df_expanded[df_expanded["exchange"] == exchange],
                "trading_pair": trading_pair,
            }
        )


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

first_target = 0.005
second_target = 0.01
exchange = "okx"
trading_pair = "USDT-BRL"

data = [data for data in exchange_data[exchange] if data["trading_pair"] == trading_pair][0]
fig = go.Figure()
candles_trace = go.Candlestick(x=data["candles_df"]["date"],
                               open=data["candles_df"]["open"],
                               high=data["candles_df"]["high"],
                               low=data["candles_df"]["low"],
                               close=data["candles_df"]["close"])
daily_volume_df = data["candles_df"].groupby("date")["volume"].sum().reset_index()
bot_daily_volume = data["activity"].groupby("date")["total_usdt_volume"].sum().reset_index()

In [ ]:
overall_volume_df = daily_volume_df.merge(bot_daily_volume, on="date", how="left")
overall_volume_df["target_0.01"] = overall_volume_df["volume"] * 0.01
overall_volume_df["market_participation"] = overall_volume_df["total_usdt_volume"] / overall_volume_df["volume"]

fig = go.Figure()

# Left axis (absolute volumes)
fig.add_trace(
    go.Bar(
        name="Daily 1% Target",
        x=overall_volume_df["date"],
        y=overall_volume_df["target_0.01"]
    )
)

fig.add_trace(
    go.Bar(
        name="Bot Daily Volume",
        x=overall_volume_df["date"],
        y=overall_volume_df["total_usdt_volume"],
        marker_color="lime"
    )
)

# Right axis (percentages)
fig.add_trace(
    go.Scatter(
        name="Total Market Share",
        x=overall_volume_df["date"],
        y=overall_volume_df["market_participation"],
        mode="lines+markers",
        yaxis="y2",  # 👈 put this trace on the right axis
        line=dict(color="white", width=2)
    )
)

# Configure axes
fig.update_layout(
    yaxis=dict(
        title="Volume (USDT)"
    ),
    yaxis2=dict(
        title="Market Share (%)",
        overlaying="y",       # share same x
        side="right",
        tickformat=".2%"      # format as percentage
    ),
    barmode="group",
    height=800
)

# Print summary
total_bot_volume = overall_volume_df["total_usdt_volume"].sum()
print(f"Overall Bot Volume (USDT): {total_bot_volume}")

fig.show()


In [ ]:
stats_df

In [ ]:
# TODO
# 1) Transform all stats_df["config"] dicts into new stats_df columns
# 2) Parse fields from columns. For floats, use astype(float) cause there are some strings. For lists calculate min, max, avg spread and n_levels renaming with suffixes, then drop original column
# 3) Calculate pnl_usdt / total_volume_usdt as new column
# 4) Concatenate into a one big df subsetting by all_cols
# 5) Create Parallel Coordinates Plot
# 6) Develop multiple controllers


In [ ]:
# ---------------------------------------------------------------------
# Config: define columnas esperadas (puedes editar libremente)
# ---------------------------------------------------------------------

BASE_KEEP = ["total_volume_usdt", "pnl_usdt", "config_file_path"]
CONTROLLER_SPECIFIC_COLS = {
    "pmm": {
        "float": [
            "total_amount_quote", "portfolio_allocation", "target_base_pct", "min_base_pct",
            "max_base_pct", "executor_refresh_time", "cooldown_time", "max_skew",
        ],
        "list": ["buy_spreads"],
        "bool": ["tick_mode"],
        "string": ["controller_name", "connector_name", "trading_pair"],
    },
    "pmm_mister": {
        "float": [
            "buy_cooldown_time", "sell_cooldown_time", "buy_position_effectivization_time",
            "sell_position_effectivization_time", "min_buy_price_distance_pct",
            "min_sell_price_distance_pct", "breakeven_buffer_pct", "dynamic_cooldown_multiplier",
            "max_active_executors_by_level",
        ],
        "list": ["buy_spreads"],
        "bool": ["tick_mode"],
        "string": ["controller_name", "connector_name", "trading_pair"]
    }
}


In [ ]:
def normalize_config(df: pd.DataFrame, col: str = "config") -> pd.DataFrame:
    """Aplana el dict de 'config' en columnas nuevas."""
    cfg = pd.json_normalize(df[col].fillna({}))
    cfg.index = df.index  # alinea por índice
    return pd.concat([df.drop(columns=[col]), cfg], axis=1)

def flatter(list_of_lists: list) -> list:
    flat = []
    for item in list_of_lists:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)
    return flat

plain_stats_df = normalize_config(stats_df)
controller_names = list(plain_stats_df["controller_name"].unique())
controller_names

In [ ]:
controller_name = "pmm_mister"

plain_stats_df_filtered = plain_stats_df[plain_stats_df["controller_name"] == controller_name]
controller_columns = flatter(BASE_KEEP + [columns for dtype, columns in CONTROLLER_SPECIFIC_COLS[controller_name].items()])
controller_df = plain_stats_df_filtered[controller_columns].copy()
controller_df = controller_df.loc[:, ~controller_df.columns.duplicated(keep=False)]

for col in CONTROLLER_SPECIFIC_COLS[controller_name]["list"]:
    controller_df.loc[:, col + "_max"] = controller_df[col].apply(lambda x: max(x))
    controller_df.loc[:, col + "_n_levels"] = controller_df[col].apply(lambda x: len(x))
    controller_df.loc[:, col + "_avg_spread"] = controller_df[col].apply(lambda x: float(np.nanmean(np.diff(x))))
    controller_df.drop(columns=[col], inplace=True)

for col in CONTROLLER_SPECIFIC_COLS[controller_name]["float"]:
    controller_df[col] = controller_df[col].astype(float)

controller_df["pnl_volume_ratio"] = controller_df["pnl_usdt"] / controller_df["total_volume_usdt"]

fig = px.parallel_coordinates(controller_df, color="pnl_volume_ratio", dimensions=controller_df.columns)
fig.show()

In [ ]:
dimensions = []

In [ ]:

fig.show()